## 🔍 Welding Defect Detection (PyTorch)

This project uses a fine-tuned ResNet18 model to classify welding images as either `defect` or `no_defect`. It leverages PyTorch and transfer learning for fast training and accurate results, even on small datasets.

### ✅ Features
- Binary classification using pretrained ResNet18
- Compatible with CPU and GPU
- Folder-based dataset loading with `ImageFolder`
- Outputs precision, recall, and F1-score

### 📁 Dataset
You can use this welding defect dataset from Kaggle:  
[Weld Defect Detection Dataset](https://www.kaggle.com/datasets/sukmaadhiwijaya/weld-defect-detection-dataset)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import os

In [ ]:
# Set parameters
batch_size = 16
epochs = 5
learning_rate = 0.001
image_size = 224

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

In [ ]:
# Load datasets
train_path = 'data/train'
val_path = 'data/val'

train_dataset = datasets.ImageFolder(root=train_path, transform=transform)
val_dataset = datasets.ImageFolder(root=val_path, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [ ]:
# Define model
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, len(train_dataset.classes))
model = model.to(device)

In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [ ]:
# Training loop
print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

In [ ]:
# Evaluation
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=val_dataset.classes))